<a href="https://colab.research.google.com/github/chriski777/mech_interp/blob/main/ji_an_2025_small_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Neurofeedback Task

This notebook tries to reproduce and build off of Ji-An et al. (2025), *Language Models Are Capable of Metacognitive Monitoring and Control of Their Internal Activations* ([arXiv:2505.13763](https://arxiv.org/abs/2505.13763)) on a smaller model that we can run off of Colab GPU.



## Setup

In [ ]:
%pip install -q "transformer_lens>=4.0,<5" scikit-learn

### Initialize Settings

Import relevant packages

In [ ]:
import importlib.metadata as md
import transformers, torch
import os
import numpy as np
import pandas as pd
import yaml
import time

from transformers import AutoModelForCausalLM
from google.colab import userdata
from packaging.version import Version
from tqdm.auto import tqdm # Import this just for progress bar


installed = Version(md.version("transformers"))
if Version(transformers.__version__) != installed:
    raise RuntimeError("An old copy of transformers is still loaded. "
                       "Runtime → Restart session, then run from this cell.")
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU."
print("transformers", installed, "| transformer_lens", md.version("transformer_lens"),
      "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---- Settings ---------------------------------------------------------------
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # Model name to fetch
LAYERS = [0, 3, 7, 11, 15]     # the paper's layers for this model (0/25/50/75/100% depth)
REPORT_LAYERS = [7]

DTYPE = torch.float16
AXES = ["LR", "LDA", "two_step_PCA_LR", "two_step_PCA_LDA", "PC1", "PC4", "PC32", "PC128", "PC512"]

RUN_SHUFFLED_CONTROL = True

N_EXAMPLES = 100               # labeled sentences per prompt (paper: 500)
N_REPEATS = 10                 # random sentence samples per condition (paper: 100)
SEED = 42
DATE_STRING = "26 Jul 2024"    #

USE_DRIVE = True               # save results to Google Drive so a disconnect doesn't lose them

Load in hugging face access token and initialize directory for saving results

In [ ]:
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/neurofeedback_llama1b"
else:
    SAVE_DIR = "/content/neurofeedback_llama1b"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to", SAVE_DIR)

## Load dataset

We use the ETHICS commonsense-morality sentences and the prompt text saved in the authors' repository.


### Download dataset

In [ ]:
# Uncomment the code below to copy the sentences and prompts from the author


#!git clone -q --depth 1 https://github.com/sakimarquis/llm_neurofeedback.git /content/llm_neurofeedback 2>/dev/null || echo "Repo already cloned."
# import shutil
# from pathlib import Path

# SRC = Path(REPO)        # /content/llm_neurofeedback
# DST = Path(SAVE_DIR)

# FILES = [
#     "data/ethics_commonsense/cm_train.csv",
#     "data/ethics_commonsense/cm_test.csv",
#     "configs/prompts.yml",
#     "configs/meta_prompts.yml",
# ]

# for rel in FILES:
#     src, dst = SRC / rel, DST / rel
#     dst.parent.mkdir(parents=True, exist_ok=True)
#     shutil.copy2(src, dst)
#     print(f"Copied {rel} -> {dst}")

### Load sentences and prompts

Following the authors' `load_commonsense`, we use only the short scenarios (one or two sentences each) and interleave morally acceptable (label 0) and unacceptable (label 1) examples.

We select `n_sample = 10000` sentences. We use 60% of the samples to  fit the axes and 40% for reporting.  Like the authors, we don't shuffle within labels, so the selection of train/test samples is deterministic.

In [ ]:
DATA_DIR = SAVE_DIR

def load_commonsense(data_dir, n_sample=10000, train_ratio=0.6):
    # Mirrors data/load.py::load_commonsense in the authors' repo.
    df = pd.concat([pd.read_csv(f"{data_dir}/data/ethics_commonsense/cm_train.csv"),
                    pd.read_csv(f"{data_dir}/data/ethics_commonsense/cm_test.csv")], ignore_index=True)
    df = df[df["is_short"] == True]
    zeros, ones = df[df["label"] == 0], df[df["label"] == 1]
    n = min(len(zeros), len(ones))
    rows = [r for pair in zip(zeros.iloc[:n].itertuples(), ones.iloc[:n].itertuples()) for r in pair]
    rows = rows[:n_sample]
    sentences = [r.input.strip() for r in rows]
    labels = np.array([r.label for r in rows])
    split = int(len(rows) * train_ratio)  # use actual number of rows
    return sentences[:split], labels[:split], sentences[split:], labels[split:]

train_sents, train_moral, test_sents, test_moral = load_commonsense(DATA_DIR)

# Just making sure that the moral scores are not in any of the the sentences
assert not any("[Score:" in s for s in train_sents + test_sents)

# Load all the prompts and meta prompts
with open(f"{DATA_DIR}/configs/prompts.yml") as f:
    prompts = yaml.safe_load(f)
with open(f"{DATA_DIR}/configs/meta_prompts.yml") as f:
    meta_prompts = yaml.safe_load(f)

SYSTEM_MSG, USER_MSG = prompts["system"], prompts["user_msg"]
REPORT_SYSTEM = (meta_prompts["report_NF"][0]["content"]
                 .replace("<SCORE_SET>", "0, 1").replace("<LOWEST>", "0").replace("<HIGHEST>", "1"))

print(f"{len(train_sents)} axis-fitting sentences, {len(test_sents)} reporting sentences")

example_idx = np.random.randint(len(train_sents))
print("\nExample sentence. :", train_sents[example_idx], 'Score:', train_moral[example_idx])
print("\nReporting-task system prompt:\n", REPORT_SYSTEM)

### Helper functions for tokenizing sentences

In [ ]:
from bisect import bisect_left, bisect_right
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def render(messages):
    # Remove date string templates
    return tokenizer.apply_chat_template(messages, tokenize=False, date_string=DATE_STRING)

def _n_tokens(text):
    # count number of tokens
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def locate(text, char_spans):
    # char_spans is the positions in the text string
    # (each index is a tuple of the character start and character end)
    # enc has token_ids, attention_mask (all 1's; don't matter), offset_mapping (start,end of text)  for each token
    # with add_special_tokens set to false, we don't want the BOS token added.
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    # Get the start position of each token
    starts = [s for s, _ in enc["offset_mapping"]]

    def same(a, b):
        return " ".join(a.split()) == " ".join(b.split())

    def ok(spans):
        return all(same(tokenizer.decode(ids[s:e], clean_up_tokenization_spaces=False), text[cs:ce])
                   for (s, e), (cs, ce) in zip(spans, char_spans))
    spans = [(bisect_right(starts, cs) - 1, bisect_left(starts, ce)) for cs, ce in char_spans]
    if not ok(spans):
        spans = [(_n_tokens(text[:cs]), _n_tokens(text[:ce])) for cs, ce in char_spans]
        if not ok(spans):
            raise ValueError("Could not map text spans to tokens for this tokenizer.")
    return ids, spans

# Quick check on one sentence
_t = render([{"role": "system", "content": SYSTEM_MSG}, {"role": "user", "content": USER_MSG},
             {"role": "assistant", "content": test_sents[0]}])
# Retrieve character span indices
_cs = _t.rfind(test_sents[0])
# Convert into token indices
_ids, [(_s, _e)] = locate(_t, [(_cs, _cs + len(test_sents[0]))])
print(repr(_t))
print("Pooled tokens:", tokenizer.convert_ids_to_tokens(_ids[_s:_e])) # text stored as bytes

## Record activations using TransformerLens

Here we use transformerLens model' wrapper to load the downloaded model's weights.


In [ ]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(MODEL_NAME, device="cuda", dtype=DTYPE)
N_LAYERS, D_MODEL = bridge.cfg.n_layers, bridge.cfg.d_model # (number of transformer blocks, width of the residual stream)
assert max(LAYERS) < N_LAYERS
print(f"{MODEL_NAME}: {N_LAYERS} layers, residual stream width {D_MODEL}")

In [ ]:
HOOKS = [f"blocks.{l}.hook_out" for l in LAYERS]

@torch.inference_mode()
def record(sentences, desc):
    """
    Takes in a list of sentences and a label for the progress bar. Outputs the activiations for each sentence
    """
    # initialize dictionary that contains activations corresponding to each sentence at each layer {layer_num: d_model}
    activations = {l: np.zeros((len(sentences), D_MODEL), dtype=np.float32) for l in LAYERS}
    # sets the progress bar
    for i, sent in enumerate(tqdm(sentences, desc=desc)):
        # apply model chat template to turn into chat prompt
        text = render([{"role": "system", "content": SYSTEM_MSG},
                       {"role": "user", "content": USER_MSG},
                       {"role": "assistant", "content": sent}])
        # find start of text within the chat prompt
        sent_start_idx_ip = text.rfind(sent)
        # find in the prompt where the sentences are and returns token ids for all of the prompt
        ids, [(s, e)] = locate(text, [(sent_start_idx_ip, sent_start_idx_ip + len(sent))])
        # Run the forward pass through the model and get the activations at the specific HOOKS
        _, cache = bridge.run_with_cache(torch.tensor([ids], device="cuda"), names_filter=HOOKS)
        # Fill in the activations based on the forward pass
        for l, h in zip(LAYERS, HOOKS):
            # Save the mean activations observed for the sentence
            activations[l][i] = cache[h][0, s:e].float().mean(0).cpu().numpy()
    return activations


In [ ]:
create_new_activ_file = True # Set to false if don't want to make new activations


activation_file = f"{SAVE_DIR}/activations_{MODEL_NAME.split('/')[-1]}.npz"

if (os.path.exists(activation_file)) and not(create_new_activ_file):
    saved = np.load(activation_file)
    acts_train = {l: saved[f"train_L{l}"] for l in LAYERS}
    acts_test = {l: saved[f"test_L{l}"] for l in LAYERS}
    print("Loaded saved activations from", activation_file)
    print("")
else:
    acts_train = record(train_sents, "Extracting activations for axis-fitting set")
    acts_test = record(test_sents, "Extracting activations for actual reporting trials")
    np.savez(activation_file, **{f"train_L{l}": acts_train[l] for l in LAYERS},
                       **{f"test_L{l}": acts_test[l] for l in LAYERS})
    print("Saved activations to", activation_file)

n_train = next(iter(acts_train.values())).shape[0]
n_test = next(iter(acts_test.values())).shape[0]
print(f"Activations loaded for : {n_train} train sentences, {n_test} test sentences")

## Define axes that will be used for feedback control


In [ ]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
pcs_needed = sorted(int(a[2:]) for a in AXES if a.startswith("PC"))
projs, axes_info = {}, {}

for l in LAYERS:
    Xtr, Xte = acts_train[l], acts_test[l]

    # Fit LR model on z-scored values
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, train_moral)

    # Fit PCA
    pca = PCA(n_components=min(Xtr.shape) - 1).fit(Xtr)

    # Fit linear discriminant analysis on z-scored values
    lda = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()).fit(Xtr, train_moral)

    # Fit two-step PCA (95% variance) + LR
    pca_lr = make_pipeline(StandardScaler(), PCA(n_components=0.95), LogisticRegression(max_iter=5000)).fit(Xtr, train_moral)

    # Fit two-step PCA (95% variance) + LDA
    pca_lda = make_pipeline(StandardScaler(), PCA(n_components=0.95), LinearDiscriminantAnalysis()).fit(Xtr, train_moral)

    n_pca_dims = pca_lr.named_steps["pca"].n_components_

    # Fit a Mean difference axis
    mean0 = Xtr[train_moral == 0].mean(axis=0)
    mean1 = Xtr[train_moral == 1].mean(axis=0)
    mean_diff = mean1 - mean0
    mean_diff_norm = mean_diff / np.linalg.norm(mean_diff)

    # Compute projections
    projs[l] = {
        "LR": lr.decision_function(Xte),
        "LDA": lda.decision_function(Xte),
        "two_step_PCA_LR": pca_lr.decision_function(Xte),
        "two_step_PCA_LDA": pca_lda.decision_function(Xte),
        "mean_diff":(Xte - mean0) @ mean_diff_norm,
    }

    # Compute the projections for each of the PCs
    for k in pcs_needed:
        projs[l][f"PC{k}"] = (Xte - pca.mean_) @ pca.components_[k - 1]

    scaler, lr_clf = lr.named_steps["standardscaler"], lr.named_steps["logisticregression"]
    lda_clf = lda.named_steps["lineardiscriminantanalysis"]
    axes_info[l] = dict(lr_w=lr_clf.coef_[0],
                        lr_b=lr_clf.intercept_[0],
                        train_mean=scaler.mean_,
                        train_std=scaler.scale_,
                        lda_w=lda_clf.coef_[0],
                        lda_b=lda_clf.intercept_[0],
                        mean_diff=mean_diff_norm,
                        mean_diff_center=mean0,
                        pca_mean=pca.mean_,
                        pca_components=pca.components_[:max(pcs_needed)],
                        pca_var_ratio=pca.explained_variance_ratio_[:max(pcs_needed)]
                  )

    print(f"Layer {l:>2}: LR morality accuracy on held-out sentences = {lr.score(Xte, test_moral):.2f}")
    print(f"Layer {l:>2}: LDA morality accuracy on held-out sentences = {lda.score(Xte, test_moral):.2f}")
    print(f"Layer {l:>2}: two_step_PCA_LR morality accuracy on held-out sentences = {pca_lr.score(Xte, test_moral):.2f} ({n_pca_dims} PCA dims)")
    print(f"Layer {l:>2}: two_step_PCA_LDA morality accuracy on held-out sentences = {pca_lda.score(Xte, test_moral):.2f}")


In [ ]:
# Convert the projections of test sentences into labels based on the sign of their projection
labels = {l: {a: (s >= 0).astype(int) for a, s in projs[l].items()} for l in LAYERS}

# Save into npz
proj_output_filename = f"{SAVE_DIR}/axes_{MODEL_NAME.split('/')[-1]}.npz"
np.savez(proj_output_filename, **{f"L{l}_{k}": v for l in LAYERS for k, v in axes_info[l].items()})
print('Saved axes definitions to:', proj_output_filename)

In [ ]:
# Free GPU memory
import gc

if 'bridge' in dir():
    del bridge
gc.collect()
torch.cuda.empty_cache()

## Reporting task


We use `transformers` here instead of `TransformerLens`, which needs memory proportional to the prompt length squared (too much to fit into the T4 implementation).


The task consists of the transformer being given a prompt containing example pairs of a sentence and a label, which is based on the side of a fitted axis it lies on. We then give it a sentence without a score (denoted as ?) and ask it to guess whether it is a 0 or 1.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, attn_implementation="sdpa").to("cuda").eval()

# Get tokens for axis report labels 0 and 1
ID0, ID1 = (tokenizer.encode(d, add_special_tokens=False) for d in "01")

ID0, ID1 = (tokenizer.encode(d, add_special_tokens=False) for d in "01")
assert len(ID0) == len(ID1) == 1, "Expected '0' and '1' to be single tokens."
ID0, ID1 = ID0[0], ID1[0]

MARKER = "[Score: "


In [ ]:
# Helper functions

# Generate report prompt that will be used for the models
def build_report_prompt(sentences, labels_shown):
    msgs = [{"role": "system", "content": REPORT_SYSTEM}]
    for s, y in zip(sentences, labels_shown):
        msgs.append({"role": "user", "content": USER_MSG + "\n"})
        msgs.append({"role": "assistant", "content": f"{s} {MARKER}{y}]\n"})
    text = render(msgs)
    label_chars, pos = [], 0
    for _ in labels_shown:
        pos = text.index(MARKER, pos) + len(MARKER)
        label_chars.append(pos)
    # return the rendered text along with the character positions within text referring to 0 or 1
    return text, label_chars

@torch.inference_mode()
def report_trial(sentences, labels_shown):
    # Run forward pass to get P(label = 1) at every label position using the preceding context.
    # Get the prompt string and character position of label digit
    text, label_chars = build_report_prompt(sentences, labels_shown)
    # prompt is tokenized into ids and character positions are retrieved. Each (c, c+1) is a one-character span (0 or 1).
    ids, spans = locate(text, [(c, c + 1) for c in label_chars])
    # Start of label token
    label_tok = [s for s, _ in spans]
    # Figure out what the model is saying based on the token right before the label
    predict_at = torch.tensor([t - 1 for t in label_tok], device="cuda")
    # Run forward pass through model on full prompt and return logits right at predict_at
    logits = model(torch.tensor([ids], device="cuda"), logits_to_keep=predict_at, use_cache=False).logits[0]
    # How confident is the model that it's the label 1?
    p1 = logits[:, [ID0, ID1]].float().softmax(-1)[:, 1]
    return p1.cpu().numpy(), len(ids)

Now we have the model report at the report layer what it thinks the axis-based label is.

In [ ]:
results_file = f"{SAVE_DIR}/report_results_{MODEL_NAME.split('/')[-1]}_N{N_EXAMPLES}.csv"
done = set()

# load it if it exists
if os.path.exists(results_file):
    prev = pd.read_csv(results_file)
    # Skip trials we've already done
    done = set(prev[["layer", "axis", "repeat", "flip"]].drop_duplicates().itertuples(index=False, name=None))
    print(f"Resuming: {len(done)} trials already saved.")

# for each axis, report at the given report_layers
conditions = [(l, a) for l in REPORT_LAYERS for a in AXES]
if RUN_SHUFFLED_CONTROL:
    conditions.append((REPORT_LAYERS[0], "SHUFFLED"))
n_total = len(conditions) * N_REPEATS * 2
timing_shown = False

with tqdm(total=n_total, desc="Reporting trials") as bar:
    for rep in range(N_REPEATS):
        # randomly sample examples sentences from test set
        idx = np.random.default_rng(SEED + rep).choice(len(test_sents), N_EXAMPLES, replace=False)
        sents = [test_sents[i] for i in idx]
        for layer, axis in conditions:
            if axis == "SHUFFLED":
                # Permute the logistic regression labels
                true = np.random.default_rng(10_000 + SEED + rep).permutation(labels[layer]["LR"][idx])
            else:
                # Get the true axis label based on the current saved
                true = labels[layer][axis][idx]
            for flip in (False, True):
                bar.update(1)
                if (layer, axis, rep, flip) in done:
                    continue
                shown = 1 - true if flip else true
                t0 = time.time()
                try:
                    p1, n_tok = report_trial(sents, shown)
                except torch.cuda.OutOfMemoryError as err:
                    raise RuntimeError("GPU out of memory. Lower N_EXAMPLES and re-run this cell.") from err
                if not timing_shown:
                    bar.write(f"Each prompt is ~{n_tok:,} tokens; one trial took {time.time() - t0:.1f} s.")
                    timing_shown = True
                # Position is for sentence within the prompt (how many has the model seen so far)
                pd.DataFrame({"layer": layer, "axis": axis, "repeat": rep, "flip": flip,
                              "position": np.arange(N_EXAMPLES), "label": shown, "p1": p1}
                             ).to_csv(results_file, mode="a", header=not os.path.exists(results_file), index=False)
print("Results saved to", results_file)

In [ ]:
df.head(5)

In [ ]:
import matplotlib.pyplot as plt

# Initialize colors for plots
INK, INK_2, MUTED, GRID, BASELINE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
LR_COLOR = "#eb6834"
LDA_COLOR = "#6a3d9a"
PCA_LR_COLOR = "#2ca02c"
PCA_LDA_COLOR = "#98df8a"
MEAN_DIFF_COLOR = "#e377c2"
PC_RAMP = ["#0d366b", "#1c5cab", "#2a78d6", "#5598e7", "#86b6ef"]   # darker = higher-variance PC
BIN = 10

